# Attendance IV: data preparation and first stage

This notebook reproduces the data-cleaning and instrument-relevance work for the ECON521 project. The MLB and NBA are the analysis leagues; the NFL is used only to calibrate a common close-game percentile. NHL is excluded because a suitable dataset was not available.

Current-game MLB/NBA analyses use calendar years 2001–2019, matching the supplied NFL file's observed years. Cleaned files preserve the full source history, and previous-game links are built before restricting the current-game sample.

The proposed excluded instruments are indicators that the current home team's and current away team's immediately previous games were close. No home-win outcome regression or second-stage IV estimate is run here. The exclusion restriction remains an open concern because previous-game closeness may affect later performance directly.

In [ ]:
from pathlib import Path
import subprocess
import sys
import pandas as pd
from IPython.display import display

# Find the repository root whether Jupyter starts in the repo or notebooks/.
_here = Path.cwd().resolve()
ROOT = next((p for p in [_here, *_here.parents] if (p / 'scripts' / 'prepare_data.py').exists()), None)
if ROOT is None:
    raise FileNotFoundError('Open this notebook from inside the ECON521-Project repository.')
print(f'Repository: {ROOT}')

## Rebuild the cleaned files

The raw CSVs are preserved under `Data/raw/`. This step writes canonical game-level CSVs, cleaning summaries, and same-season prior-game histories.

In [ ]:
subprocess.run([sys.executable, str(ROOT / 'scripts' / 'prepare_data.py')], cwd=ROOT, check=True)

## Re-estimate the first stage

The NFL reference window requested for this project is 2000–2023. The supplied NFL file actually covers seasons 2001–2019. The anchor is the share of available NFL games with an absolute score margin of at most 3 points. Current-game MLB/NBA samples are restricted to calendar years 2001–2019. The analysis tests the exact NFL percentile, every integer percentile from 1 through 99, and every percentile at which either league's discrete margin cutoff changes. It then reports whether any common or league-specific threshold meets both F > 10 and the 5% clustered-significance screen.

In [ ]:
subprocess.run([sys.executable, str(ROOT / 'scripts' / 'first_stage.py')], cwd=ROOT, check=True)

## Inspect the calibration and threshold search

In [ ]:
RESULTS = ROOT / 'results' / 'first_stage'
nfl_reference = pd.read_csv(RESULTS / 'nfl_reference.csv')
nfl_seasons = pd.read_csv(RESULTS / 'nfl_season_coverage.csv')
print('NFL reference coverage and anchor:')
display(nfl_reference.T.rename(columns={0: 'value'}))
display(nfl_seasons)

In [ ]:
grid = pd.read_csv(RESULTS / 'common_percentile_grid.csv')
primary = grid.loc[grid['scope'].eq('2001_2019')].copy()
max_rows = primary.loc[primary.groupby('league')['joint_F_classical'].idxmax(), [
    'league', 'percentile', 'margin_cutoff', 'joint_F_classical', 'joint_p_home_team_cluster'
]].sort_values('league')
print('Largest conventional joint F by league across the tested grid:')
display(max_rows)
print('All tested percentiles:')
display(primary[[
    'league', 'percentile', 'margin_cutoff', 'actual_game_share_at_or_below_cutoff',
    'joint_F_classical', 'joint_F_home_team_cluster', 'joint_p_home_team_cluster'
]].sort_values(['percentile', 'league']).reset_index(drop=True))

In [ ]:
selected = pd.read_csv(RESULTS / 'selected_thresholds.csv')
first_stage = pd.read_csv(RESULTS / 'selected_first_stage.csv')
print('Selected sample thresholds (NFL anchor retained when no threshold passed):')
display(selected)
print('First-stage results at the retained NFL anchor:')
display(first_stage[[
    'league', 'selected_percentile', 'selected_margin_cutoff', 'n_first_stage_observations',
    'home_prev_close_coef', 'away_prev_close_coef', 'joint_F_classical',
    'joint_F_home_team_cluster', 'joint_p_home_team_cluster', 'selected_threshold_status'
]])

In [ ]:
cleaning = ROOT / 'results' / 'cleaning'
print('Cleaning coverage:')
display(pd.read_csv(cleaning / 'data_coverage.csv'))
print('Attendance quality:')
display(pd.read_csv(cleaning / 'attendance_quality.csv'))
print('Prior-history availability:')
display(pd.read_csv(cleaning / 'history_quality.csv'))

## Interpretation

A first-stage F below 10 indicates weak instrument relevance under the requested screen. The threshold sweep covers every integer percentile from 1 through 99 and each discrete cutoff transition in the 2001–2019 margin distributions. The first stage does not establish the exclusion restriction. No second-stage analysis is included.